<a href="https://colab.research.google.com/github/BeaverWorksMedlytics2020/Week2/blob/master/Notebooks/05_NeuralNetworks/NeuralNetworks_Tutorial_part1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Building a Simple Neural Network with Pytorch

In this notebook we are going to walk through building a simple neural network to classify sequence data. This tutorial will be meant as a fast overview of building/training neural networks with Pytorch.

In [ ]:
# Import useful libraries

#Needed for terminal functions (i.e. wget)
import os

#For plotting
import matplotlib.pyplot as plt

#For dataframe manipulation
import pandas as pd
import numpy as np

#For data preprocessing
from sklearn.preprocessing import StandardScaler #Use StandardScaler from scikitlearn
from sklearn.utils import shuffle #Used to shuffle up examples before training

#Pytorch imports
import torch
import torch.nn as nn
import torch.optim as optim

In [ ]:
#Checks if dataset is in working directory, else downloads it
if not os.path.exists("spoken_digit_manual_features.csv"):
    os.system('curl -o spoken_digit_manual_features.csv https://raw.githubusercontent.com/BeaverWorksMedlytics2020/Data_Public/master/NotebookExampleData/Week2/spoken_digit_manual_features.csv')

## Load Training Data

In [ ]:
#Load dataframe and print its contents to jog memory
spoken_df = pd.read_csv('spoken_digit_manual_features.csv', index_col = 0)
print(spoken_df.head(10))
print('\n')

#Check how many unique speakers exist in the dataset
speakers=set(spoken_df['speaker'])
print(f'There are {len(speakers)} unique speakers in the dataset')

# Our goal for this is to build a neural network that learns to classify which
# of 5 speakers is recorded in a sample based on the features:
# spectral centroid, spectral flatness, and maximum frequency


                file  digit   speaker  trial           SC        SF  \
0   5_yweweler_8.wav      5  yweweler      8  1029.497959  0.397336   
1    3_george_49.wav      3    george      4  1881.296834  0.387050   
2  9_yweweler_44.wav      9  yweweler      4  1093.951856  0.394981   
3  8_yweweler_33.wav      8  yweweler      3  1409.543285  0.487496   
4      7_theo_34.wav      7      theo      3   887.361601  0.396825   
5   1_jackson_45.wav      1   jackson      4  1007.568129  0.324100   
6  6_yweweler_18.wav      6  yweweler      1  1286.701352  0.498813   
7    9_george_35.wav      9    george      3  1405.092061  0.353083   
8   9_jackson_32.wav      9   jackson      3  1172.899961  0.477907   
9    8_george_26.wav      8    george      2  1959.977577  0.462901   

           MF  
0  745.878340  
1  323.943662  
2  244.648318  
3  392.350401  
4  130.640309  
5  216.306156  
6  400.715564  
7  447.239693  
8  114.892780  
9  320.537966  


There are 5 unique speakers in the datas

## Pytorch layers

In pytorch operations such as linear layers, activation functions, etc are implemented as subclasses within the `torch.nn` superclass.
For instance: `torch.nn.ReLU()` will initialize a ReLU() object to be used as part of your model.

Alternatively one can also make use of the **functional API** if only interested in getting the output using a certain function.
For instance: `torch.nn.functional.relu(input)` won't initialize a layer, but instead just returns the output of passing your input through a ReLU function.

In [ ]:
# By convention, the first dimension will always be batch size
example_input = torch.randn(64, 5)

# nn.Linear expects arguments input_shape, output_shape
example_linear_layer = nn.Linear(5, 3)

# example usage
example_output = example_linear_layer(example_input)
print(example_output.shape)

# Now using the functional API
example_weight = torch.randn(3, 5)
example_output = torch.nn.functional.linear(
    example_input, example_weight
)
print(example_output.shape)


torch.Size([64, 3])
torch.Size([64, 3])


> #### Participation question:
> In the following example, what are the shapes at (1), (2), and (3)?
>
>```Python
>    input = torch.randn(128, 16) # (1)
>    linear1 = torch.nn.Linear(16, 32)
>    linear2 = torch.nn.Linear(32,8)
>    
>    output = linear1(input) # (2)
>    output = linear2(output) # (3)
>```
><details>
>  <summary>Answer</summary>
>  
>  (128, 16)
>
>  (128, 32)
>
>  (128, 8)
>  
></details>

## Pytorch networks(modules)

There are many ways to build a neural network in Pytorch. The easiest way is using the **Sequential API**. See [here](https://docs.pytorch.org/docs/stable/generated/torch.nn.Sequential.html) for moer info.

In [ ]:
# Building the Pytorch network

# Method 1: using nn.Sequential
# useful for quick and dirty experiments
model = nn.Sequential(
    nn.Linear(3, 8),
    nn.ReLU(),

    *[nn.Linear(8, 8),
    nn.ReLU()]*2,

    nn.Linear(8, 5),
)


More complex models with many moving parts will require you to make your own class using the **Module API** instead. See [here](https://docs.pytorch.org/docs/stable/generated/torch.nn.Module.html) for more info.

Here is an example

In [ ]:
import torch.nn.functional as f

class MyCustomModel(nn.Module):
    def __init__(self, num_hidden_layers, in_size, hidden_size, out_size):
        super().__init__()
        self.input_layer = nn.Linear(in_size, hidden_size)

        self.hidden_layers = nn.ModuleList([
            nn.Linear(hidden_size, hidden_size) for _ in range(num_hidden_layers)
        ])

        self.output_layer = nn.Linear(hidden_size, out_size)

    def forward(self, x):
        out = f.relu(self.input_layer(x))

        for hidden_layer in self.hidden_layers:
            out = f.relu(hidden_layer(out))

        out = self.output_layer(out)

        return out

model = MyCustomModel(2, 3, 8, 5)
print(model(torch.randn(64, 3)).shape)

torch.Size([64, 5])


## Defining our loss and optimizer

Let's describe why each of these components is necessary, and how it is used in training a neural network.

**Loss Function** - Loss functions are mathematical formulas the model uses to measure how well it is doing at fitting to the data. The two most basic loss functions you will encounter are *mean squared error loss* which is used in regression tasks and *cross entropy loss* which is used in classification tasks.

You may see other alternatives for `CrossEntropyLoss()` such as:
- `NLLLoss()`
- `BCELoss()`
- `BCEWithLogitLoss()`
These ALL refer to the same mathematical formula, but they differ in implementation and arguments they expect.

In [ ]:
#Specify a loss function for our network
criterion = nn.CrossEntropyLoss()

> #### Participation question:
>if mean squared error can be used to measure the difference between any two numbers such as our model's prediction and the intended target--why don't we just use that in classification as well? After all class probabilities are numbers too.
><details>
>  <summary>Answer</summary>
>  
>  Loss functions make implicit assumptions about the underlying distributions. We want our assumptions to be as close to reality as possible.
>
>  For instance:
>  - Mean squared error loss is derived by assuming `y = f(x;theta) + eps where eps ~ N(0,1)`
>  - Cross entropy loss is derived by assuming `y ~ Bernoulli(p) where p = f(x;theta)`
>
>  In a classification setting, y can only ever be 0 or 1, IE: y is a Bernoulli trial.
>  So if we would be making an incorrect assumption by using mean squared error loss
>
>  AKA: the model won't be as good :\(
>  
></details>

**Optimizer** - Optimizers define the update rule for a neural network. After we backpropagate the loss to retrieve the gradient, we still need to decide how we choose to use that information to update our network. This choice can have a massic impact on training speed and final performance. See [here](https://docs.pytorch.org/docs/stable/optim.html#algorithms) for more info.

In [ ]:
optimizer = optim.Adam(model.parameters(), lr=0.01)

## Processing label vectors

Predictions output by the model need to be compared to some truth label, currently our labels are strings of the speaker name, so we need to process that.

Pytorch can accept both one-hot encoded vectors and class index vectors for its labels. To make things simple, we'll be using class index labels.

In [ ]:
#make dictionary to convert from speaker names to indices
name2int_dict = {name: ind for (ind, name) in enumerate(set(spoken_df['speaker']))}

y_labels = spoken_df['speaker']
#set y_labels to be indices of speaker
y_labels = [name2int_dict[name] for name in y_labels]


## Standardize Data and split into train/validation/test sets

Scaling data is generally good practice before attempting to fit a model. Having inputs with large differences in scale can affect how the optimizer changes weights to minimize the loss function

In [ ]:
#downselect to only the 3 columns of the dataset we are learning from
X_data = spoken_df[['SC', 'SF', 'MF']].to_numpy()

#Decide how large to make validation and test sets
n_val = 250
n_test = 250

#Shuffle data before partitioning
X_data, y_labels = shuffle(X_data, y_labels, random_state = 25)

#Partition
X_test, y_test = X_data[:n_test,:], y_labels[:n_test]
X_val, y_val = X_data[n_test:n_test+n_val,:], y_labels[n_test:n_test+n_val]
X_train, y_train = X_data[n_test+n_val:,:], y_labels[n_test+n_val:]

#Scale data
scaler = StandardScaler()
X_train=scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

## Fit Model to Data, Specify Number of Epochs and Batch Size

**Batch Size** - In each iteration of the optimizer, how many samples are taken into account when calculating derivatives of the loss function? (If batch size is less than number of samples, there will be multiple optimization iterations per epoch.)

**Epochs** - How many times should the data be passed through before optimization is finished?

In [ ]:
epochs = 50
batch_size = 100

train = torch.utils.data.TensorDataset(
    torch.tensor(X_train, dtype=torch.float32),
    torch.tensor(y_train, dtype=torch.long)
)
train_loader = torch.utils.data.DataLoader(train, batch_size=batch_size, shuffle=False)

for epoch in range(epochs):
    for i, data in enumerate(train_loader):
        # Train loop
        model.train()
        inputs, labels = data

        # Pytorch keeps track of gradients across loops, so we need to zero it for each batch
        # We need to manually zero the gradient--reset it--every batch
        optimizer.zero_grad()

        outputs = model(inputs) # Get the model's output
        loss = criterion(outputs, labels) # Compute the loss
        preds = torch.argmax(outputs,dim=1)

        # For monitoring
        tr_loss = loss.item()
        tr_acc = torch.sum(preds==labels)/len(labels)

        loss.backward() # Backpropagate to compute gradient
        optimizer.step() # Update the parameters, according to chosen optimizer algorithm
    # Validation loop
    model.eval()
    inputs, labels = torch.tensor(X_val, dtype=torch.float32), torch.tensor(y_val, dtype=torch.long)

    outputs = model(inputs)
    preds = torch.argmax(outputs,dim=1)

    val_loss = criterion(outputs, labels).item()
    val_acc = torch.sum(preds==labels)/len(labels)

    print(('Epoch #%d\t Training Loss: %.2f\tTraining Accuracy: %.2f\t'
         'Validation Loss: %.2f\tValidation Accuracy: %.2f')
         % (epoch + 1, tr_loss, tr_acc,
         val_loss, val_acc))

Epoch #1	 Training Loss: 1.55	Training Accuracy: 0.20	Validation Loss: 1.57	Validation Accuracy: 0.22
Epoch #2	 Training Loss: 1.44	Training Accuracy: 0.32	Validation Loss: 1.41	Validation Accuracy: 0.36
Epoch #3	 Training Loss: 1.33	Training Accuracy: 0.44	Validation Loss: 1.32	Validation Accuracy: 0.40
Epoch #4	 Training Loss: 1.23	Training Accuracy: 0.49	Validation Loss: 1.24	Validation Accuracy: 0.46
Epoch #5	 Training Loss: 1.17	Training Accuracy: 0.51	Validation Loss: 1.17	Validation Accuracy: 0.46
Epoch #6	 Training Loss: 1.11	Training Accuracy: 0.54	Validation Loss: 1.12	Validation Accuracy: 0.54
Epoch #7	 Training Loss: 1.10	Training Accuracy: 0.52	Validation Loss: 1.10	Validation Accuracy: 0.56
Epoch #8	 Training Loss: 1.08	Training Accuracy: 0.51	Validation Loss: 1.10	Validation Accuracy: 0.54
Epoch #9	 Training Loss: 1.08	Training Accuracy: 0.51	Validation Loss: 1.10	Validation Accuracy: 0.54
Epoch #10	 Training Loss: 1.07	Training Accuracy: 0.52	Validation Loss: 1.09	Valid

## Check Performance on Test Set

We can use model.predict to output predicted labels on the test set, or model.evaluate to determine test-set accuracy (since we have the labels)


In [ ]:
model.eval()
inputs, labels = torch.tensor(X_val, dtype=torch.float32), torch.tensor(y_val, dtype=torch.long)

outputs = model(inputs)
preds = torch.argmax(outputs,dim=1)

test_loss = criterion(outputs, labels).item()
test_acc = torch.sum(preds==labels)/len(labels)

print(f'Test loss: {test_loss:.2f}, Test accuracy: {test_acc:.2f}')

Test loss: 1.03, Test accuracy: 0.59
